# 04 — Feature Engineering

**EduPredict — Student Exam Performance Prediction**

This notebook creates a small, evidence-driven set of engineered features from the cleaned dataset.

The features were motivated by the EDA:
- `Attendance` had a strong observed relationship with `Exam_Score`.
- `Hours_Studied` had a moderate observed relationship with `Exam_Score`.
- Their interaction showed stronger association with the target than either variable alone.

**Important:** `Exam_Score` is never used to construct input features. The engineered features use only prediction-time input variables.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
CLEAN_PATH = Path("../data/processed/student_performance_clean.csv")

df_clean = pd.read_csv(CLEAN_PATH)

print("Cleaned dataset loaded.")
print("Shape:", df_clean.shape)
display(df_clean.head())

Cleaned dataset loaded.
Shape: (6606, 20)


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [3]:
TARGET = "Exam_Score"

print("Target column:", TARGET)
print("Target missing values:", df_clean[TARGET].isnull().sum())

feature_columns = [c for c in df_clean.columns if c != TARGET]

print("Original input features:", len(feature_columns))
print("Target excluded from input features:", TARGET)

Target column: Exam_Score
Target missing values: 0
Original input features: 19
Target excluded from input features: Exam_Score


## Feature Engineering Hypotheses

### 1. Study_Attendance_Interaction

Formula:

`Hours_Studied × Attendance`

Reason:
- Study time and attendance both showed meaningful relationships with exam score during EDA.
- Their combination may capture study exposure more effectively than either variable independently.

### 2. Study_Sleep_Interaction

Formula:

`Hours_Studied × Sleep_Hours`

Reason:
- This tests whether study time combined with sleep duration contains additional predictive information.
- It is an experiment, not an assumption that sleep causes a particular score.

Both features will be evaluated later using validation performance. They are not automatically retained just because they were engineered.

In [4]:
df_fe = df_clean.copy()

df_fe["Study_Attendance_Interaction"] = (
    df_fe["Hours_Studied"] * df_fe["Attendance"]
)

df_fe["Study_Sleep_Interaction"] = (
    df_fe["Hours_Studied"] * df_fe["Sleep_Hours"]
)

print("Original columns:", len(df_clean.columns))
print("Engineered columns:", len(df_fe.columns))
print("New input features:", len(df_fe.columns) - len(df_clean.columns))

display(
    df_fe[
        [
            "Hours_Studied",
            "Attendance",
            "Sleep_Hours",
            "Study_Attendance_Interaction",
            "Study_Sleep_Interaction",
            TARGET
        ]
    ].head()
)

Original columns: 20
Engineered columns: 22
New input features: 2


,Hours_Studied,Attendance,Sleep_Hours,Study_Attendance_Interaction,Study_Sleep_Interaction,Exam_Score
0,23,84,7,1932,161,67
1,19,64,8,1216,152,61
2,24,98,7,2352,168,74
3,29,89,8,2581,232,71
4,19,92,6,1748,114,70


In [5]:
engineered_cols = [
    "Study_Attendance_Interaction",
    "Study_Sleep_Interaction"
]

print("Missing values in engineered features:")
display(df_fe[engineered_cols].isnull().sum())

print("\nInfinite values in engineered features:")
print(np.isinf(df_fe[engineered_cols]).sum())

Missing values in engineered features:


Study_Attendance_Interaction    0
Study_Sleep_Interaction         0
dtype: int64


Infinite values in engineered features:
Study_Attendance_Interaction    0
Study_Sleep_Interaction         0
dtype: int64


In [6]:
print("Engineered feature statistics:")
display(df_fe[engineered_cols].describe())

Engineered feature statistics:


,Study_Attendance_Interaction,Study_Sleep_Interaction
count,6606.000000,6606.000000
mean,1596.731759,140.501060
std,533.921040,52.496491
min,69.000000,4.000000
25%,1224.000000,104.000000
50%,1575.000000,136.000000
75%,1940.000000,174.000000
max,3783.000000,396.000000


In [7]:
comparison_cols = [
    "Hours_Studied",
    "Attendance",
    "Sleep_Hours",
    "Previous_Scores",
    "Tutoring_Sessions",
    "Physical_Activity",
    *engineered_cols,
    TARGET
]

correlation_with_target = (
    df_fe[comparison_cols]
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
    .sort_values(key=abs, ascending=False)
)

display(
    correlation_with_target.to_frame(
        "Pearson_Correlation_With_Exam_Score"
    )
)

,Pearson_Correlation_With_Exam_Score
Study_Attendance_Interaction,0.652995
Attendance,0.582458
Hours_Studied,0.446514
Study_Sleep_Interaction,0.350813
Previous_Scores,0.174461
Tutoring_Sessions,0.153754
Physical_Activity,0.027943
Sleep_Hours,-0.016194


## Interpretation of the interaction experiment

The correlation analysis is used as an exploratory signal only.

A stronger Pearson correlation does **not** by itself prove that an engineered feature improves a machine-learning model.

Final retention will be decided after:
1. feature selection,
2. validation,
3. model comparison.

This prevents arbitrary feature engineering.

In [8]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENGINEERED_PATH = (
    OUTPUT_DIR / "student_performance_engineered.csv"
)

df_fe.to_csv(ENGINEERED_PATH, index=False)

print("Engineered dataset saved to:")
print(ENGINEERED_PATH)
print("Saved shape:", df_fe.shape)

Engineered dataset saved to:
../data/processed/student_performance_engineered.csv
Saved shape: (6606, 22)


In [9]:
# Final feature inventory
input_features = [
    c for c in df_fe.columns
    if c != TARGET
]

print("Total candidate input features:", len(input_features))
print("\nCandidate features:")
for i, feature in enumerate(input_features, start=1):
    print(f"{i:02d}. {feature}")

Total candidate input features: 21

Candidate features:
01. Hours_Studied
02. Attendance
03. Parental_Involvement
04. Access_to_Resources
05. Extracurricular_Activities
06. Sleep_Hours
07. Previous_Scores
08. Motivation_Level
09. Internet_Access
10. Tutoring_Sessions
11. Family_Income
12. Teacher_Quality
13. School_Type
14. Peer_Influence
15. Physical_Activity
16. Learning_Disabilities
17. Parental_Education_Level
18. Distance_from_Home
19. Gender
20. Study_Attendance_Interaction
21. Study_Sleep_Interaction


# Feature Engineering Summary

| Feature | Formula | Purpose |
|---|---|---|
| `Study_Attendance_Interaction` | `Hours_Studied × Attendance` | Test combined study and attendance signal |
| `Study_Sleep_Interaction` | `Hours_Studied × Sleep_Hours` | Test combined study and sleep signal |

### Decision

Both engineered features are retained temporarily as **candidate features**.

They will not automatically become final features. The next phase will use the training data to perform feature selection and compare their contribution with the original features.

### Leakage Check

- `Exam_Score` is not used to construct either feature.
- Both features depend only on input variables available before prediction.
- The original raw dataset remains unchanged.
